# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://aoneahsan.com")
links

['mailto:aoneahsan@gmail.com',
 'tel:+17139134704',
 'tel:+17139134707',
 'https://wa.me/923046619706',
 'https://aoneahsan.com',
 'https://linkedin.com/in/aoneahsan',
 'https://github.com/aoneahsan',
 'https://npmjs.com/~aoneahsan',
 'https://aoneahsan.com/privacy-policy',
 'https://aoneahsan.com/terms']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://aoneahsan.com"))


Here is the list of links on the website https://aoneahsan.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

mailto:aoneahsan@gmail.com
tel:+17139134704
tel:+17139134707
https://wa.me/923046619706
https://aoneahsan.com
https://linkedin.com/in/aoneahsan
https://github.com/aoneahsan
https://npmjs.com/~aoneahsan
https://aoneahsan.com/privacy-policy
https://aoneahsan.com/terms


In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://aoneahsan.com")

{'links': [{'type': 'home page', 'url': 'https://aoneahsan.com'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://aoneahsan.com")

Selecting relevant links for https://aoneahsan.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://aoneahsan.com'},
  {'type': 'LinkedIn profile', 'url': 'https://linkedin.com/in/aoneahsan'},
  {'type': 'GitHub page', 'url': 'https://github.com/aoneahsan'},
  {'type': 'npm profile', 'url': 'https://npmjs.com/~aoneahsan'},
  {'type': 'WhatsApp contact', 'url': 'https://wa.me/923046619706'}]}

In [ ]:
select_relevant_links("https://aoneahsan.com")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://aoneahsan.com"))

Selecting relevant links for https://aoneahsan.com by calling gpt-5-nano
Found 5 relevant links
## Landing Page:

Ahsan Mahmood - Senior Full Stack Developer | Portfolio

Ahsan Mahmood - Senior Full Stack Developer | Portfolio
About Ahsan Mahmood
Ahsan Mahmood is a Senior Full Stack Developer with 8+ years of experience building scalable web
        and mobile applications. Specializing in MERN Stack (MongoDB, Express.js, React, Node.js),
        React + Firebase, TypeScript, and cross-platform mobile development with React Native and CapacitorJS.
        Based in Pakistan, available for remote work worldwide.
Technical Expertise
Frontend: React 19, Next.js, TypeScript, TailwindCSS, Radix UI
Backend: Node.js, Express.js, Firebase Cloud Functions
Database: MongoDB, Firebase Firestore, PostgreSQL
Mobile: React Native, CapacitorJS, iOS, Android
Cloud: Firebase, Google Cloud Platform, AWS
DevOps: Docker, CI/CD, GitHub Actions
UI Libraries: Radix UI, ShadcnUI, Material UI, Chakra UI
Service

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("AhsanMahmood (Aoneahsan)", "https://aoneahsan.com")

Selecting relevant links for https://aoneahsan.com by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: AhsanMahmood (Aoneahsan)\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nAhsan Mahmood - Senior Full Stack Developer | Portfolio\n\nAhsan Mahmood - Senior Full Stack Developer | Portfolio\nAbout Ahsan Mahmood\nAhsan Mahmood is a Senior Full Stack Developer with 8+ years of experience building scalable web\n        and mobile applications. Specializing in MERN Stack (MongoDB, Express.js, React, Node.js),\n        React + Firebase, TypeScript, and cross-platform mobile development with React Native and CapacitorJS.\n        Based in Pakistan, available for remote work worldwide.\nTechnical Expertise\nFrontend: React 19, Next.js, TypeScript, TailwindCSS, Radix UI\nBackend: Node.js, Express.js, Firebase Cloud Functions\nDatabase: MongoDB, Firebase Firestore, PostgreSQL\nMobile: React Native, CapacitorJS, iOS, A

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("AhsanMahmood (Aoneahsan)", "https://aoneahsan.com")

Selecting relevant links for https://aoneahsan.com by calling gpt-5-nano
Found 4 relevant links


# Ahsan Mahmood (Aoneahsan)  
**Senior Full Stack Developer | Expert in Scalable Web & Mobile Applications**  

---

## About Ahsan Mahmood  
Ahsan Mahmood is a highly skilled Senior Full Stack Developer with over 8 years of professional experience in designing and building scalable web and mobile solutions. Based in Pakistan, Ahsan is available for remote engagements worldwide. His core expertise spans modern technologies focused on fast, efficient, and scalable development — making him a versatile partner for startups, enterprises, and SaaS product teams alike.

---

## Technical Expertise  

**Frontend:**  
- React 19, Next.js, TypeScript  
- TailwindCSS, Radix UI, ShadcnUI, Material UI, Chakra UI  

**Backend:**  
- Node.js, Express.js, Firebase Cloud Functions  

**Databases:**  
- MongoDB, Firebase Firestore, PostgreSQL  

**Mobile Development:**  
- React Native, CapacitorJS, iOS, Android  

**Cloud & DevOps:**  
- Firebase, Google Cloud Platform, AWS  
- Docker, CI/CD pipelines, GitHub Actions  

---

## Services Offered  

- Full Stack Web Application Development  
- Cross-Platform Mobile App Development (iOS & Android)  
- React + Firebase Projects  
- API Development & Integration  
- Database Design & Optimization  
- Code Review & Technical Consultation  
- Legacy System Migration & Modernization  
- Progressive Web App (PWA) Development  

---

## Portfolio Highlights  

- 50+ completed projects spanning multiple industries  
- E-commerce platforms including payment gateway integration  
- Real-time applications leveraging Firebase  
- Cross-platform mobile applications with React Native and CapacitorJS  
- Enterprise dashboard and analytics solutions  
- SaaS product conception, development, and delivery  

---

## Customers & Collaborations  

Serving a diverse client base globally, including startups, growing businesses, and enterprise clients seeking sophisticated web and mobile solutions with modern tech stacks like MERN and Firebase. Projects often involve modernization, legacy system upgrades, and real-time application needs.

---

## Company Culture & Work Approach  

- Committed to clean, maintainable, and scalable code  
- Customer-centric approach: tailoring solutions to client needs  
- Emphasis on continuous learning and technology adoption  
- Transparent and responsive communication in remote collaborations  
- Passionate about quality, best practices, and code review processes  

---

## Careers & Opportunities  

Ahsan Mahmood primarily operates as a senior independent developer available for remote work engagements worldwide. While no direct hiring information is listed, potential collaborators, clients, and recruiters can connect via the contact details below to discuss contract or partnership opportunities.

---

## Contact Information  

- **Name:** Ahsan Mahmood  
- **Email:** aoneahsan@gmail.com  
- **Work Phone:** +1 713-913-4704 (USA)  
- **Texas Phone:** +1 713-913-4707 (USA)  
- **WhatsApp:** +92 304 6619706 (Pakistan)  
- **Website:** [https://aoneahsan.com](https://aoneahsan.com)  
- **LinkedIn:** [linkedin.com/in/aoneahsan](https://linkedin.com/in/aoneahsan)  
- **GitHub:** [github.com/aoneahsan](https://github.com/aoneahsan)  
- **NPM:** [npmjs.com/~aoneahsan](https://www.npmjs.com/~aoneahsan)  

---

Experience modern, scalable, and high-quality full stack development with Ahsan Mahmood — your trusted partner for web and mobile innovation.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("AhsanMahmood (Aoneahsan)", "https://aoneahsan.com")

Selecting relevant links for https://aoneahsan.com by calling gpt-5-nano
Found 1 relevant links


# Ahsan Mahmood (Aoneahsan)  
**Senior Full Stack Developer | Expert in Scalable Web & Mobile Applications**

---

## About Ahsan Mahmood  
Ahsan Mahmood is a seasoned Senior Full Stack Developer with over 8 years of professional experience specializing in delivering scalable, robust web and mobile solutions. Based in Pakistan, Ahsan is available for remote projects worldwide, bringing expert proficiency in modern technology stacks and agile development practices.

---

## Technical Expertise  
- **Frontend:** React 19, Next.js, TypeScript, TailwindCSS, Radix UI  
- **Backend:** Node.js, Express.js, Firebase Cloud Functions  
- **Database:** MongoDB, Firebase Firestore, PostgreSQL  
- **Mobile:** React Native, CapacitorJS, iOS & Android app development  
- **Cloud Platforms:** Firebase, Google Cloud Platform, AWS  
- **DevOps Tools:** Docker, CI/CD pipelines, GitHub Actions  
- **UI Libraries:** Radix UI, ShadcnUI, Material UI, Chakra UI  

---

## Services Offered  
- Full Stack Web Application Development  
- Cross-Platform Mobile App Development (iOS & Android)  
- React + Firebase Project Implementation  
- API Development & Integration  
- Database Design & Performance Optimization  
- Code Review & Technical Consultation  
- Legacy System Migration & Modernization  
- Progressive Web App (PWA) Development  

---

## Portfolio Highlights  
- Successfully completed 50+ projects spanning diverse industries  
- Developed e-commerce platforms with seamless payment integrations  
- Built real-time applications leveraging Firebase technology  
- Delivered cross-platform mobile applications with React Native & CapacitorJS  
- Created enterprise-level dashboard solutions tailored to business needs  
- Contributed to SaaS product development empowering scalable software delivery  

---

## Company Culture  
Ahsan Mahmood fosters a culture of continuous learning, innovation, and technical excellence. Emphasizing clean, maintainable code and scalable architecture, the approach blends best practices with a collaborative mindset for remote work efficiency. Strong commitment to client satisfaction, transparency, and timely delivery is core to the work ethic.

---

## Customers  
Clients range from startups to established enterprises seeking modern, scalable digital solutions. Ahsan specializes in delivering tailor-made products that optimize business workflows, enhance user experience, and leverage cutting-edge web and mobile technologies.

---

## Careers & Opportunities  
Currently, Ahsan Mahmood offers freelance and remote contract roles suitable for developers interested in collaboration or mentorship under a senior full stack developer with rich industry experience. Interested professionals can connect to explore project-based involvement or technical consultation engagements.

---

## Contact Information  
- **Name:** Ahsan Mahmood  
- **Email:** aoneahsan@gmail.com  
- **Work Phone:** +1 713-913-4704  
- **Texas Phone:** +1 713-913-4707  
- **WhatsApp:** +92 304 6619706  
- **Website:** [https://aoneahsan.com](https://aoneahsan.com)  
- **LinkedIn:** [linkedin.com/in/aoneahsan](https://linkedin.com/in/aoneahsan)  
- **GitHub:** [github.com/aoneahsan](https://github.com/aoneahsan)  
- **NPM:** [npmjs.com/~aoneahsan](https://npmjs.com/~aoneahsan)  

---

Empower your digital transformation with Ahsan Mahmood — delivering innovation, scalability, and reliability in full stack development. Reach out today to discuss your next project!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("AhsanMahmood (Aoneahsan)", "https://aoneahsan.com")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://aoneahsan.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/aoneahsan">@aoneahsan<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@aoneahsan.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>